# Deep Learning Project - 2025

**Authors:**  
- Antonio N. Bruno (ID: 258035)  
- Edoardo Di Tommaso (ID: 258433)

## Introduction

In recent years, large-scale Vision-Language Models (VLMs) such as CLIP [citare] have demonstrated remarkable zero-shot classification performance by aligning images and text in a shared embedding space. This architecture enables flexible, prompt-driven recognition without any task-specific fine-tuning. However, when applied to fine-grained classification tasks, such as distinguishing between species of flowers, birds, or aircraft, the model’s performance tends to degrade, especially in low-data regimes.

This has motivated growing interest in **few-shot adaptation**, where the goal is to adapt a pre-trained VLM to a new classification task using only a small number of labeled examples per class. In this setup, the model is trained on a few samples (shots) from a set of **base classes**, and then evaluated on both the base and a disjoint set of **novel classes**. The central challenge is to improve accuracy on base classes without sacrificing generalization to novel ones; this setting is often referred to as **base-to-novel generalization**.

The goal of our project is to develop a few-shot adaptation method which improves performance on base classes while maintaining (or even enhancing) the network's original zero-shot performance on novel classes. The VLM backbone is CLIP ViT-B/16, and the tests will be conducted on the Oxford Flowers [citare] dataset.


In [ ]:
# installing and importing required packages ...

!pip install openai-clip

import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import clip
from tqdm import tqdm

: 

## Utilities for data handling

We will now create functions to correctly get our data and split it into base and novel classes.

In [ ]:
def get_data(data_dir="./data", transform=None):
    """Load Flowers102 train, validation and test sets.
    Args:
        data_dir (str): Directory where the dataset will be stored.
        transform (torch.Compose)
    Returns:
        tuple: A tuple containing the train, validation, and test sets.
    """
    train = torchvision.datasets.Flowers102(root=data_dir, split="train", download=True, transform=transform)
    val = torchvision.datasets.Flowers102(root=data_dir, split="val", download=True, transform=transform)
    test = torchvision.datasets.Flowers102(root=data_dir, split="test", download=True, transform=transform)
    return train, val, test

def base_novel_categories(dataset):
    # set returns the unique set of all dataset classes
    all_classes = set(dataset._labels)
    # and let's count them
    num_classes = len(all_classes)

    # here list(range(num_classes)) returns a list from 0 to num_classes - 1
    # then we slice the list in half and generate base and novel category lists
    base_classes = list(range(num_classes))[:num_classes//2]
    novel_classes = list(range(num_classes))[num_classes//2:]
    return base_classes, novel_classes

def split_data(dataset, base_classes):
    # these two lists will store the sample indexes
    base_categories_samples = []
    novel_categories_samples = []

    # we create a set of base classes to compute the test below in O(1)
    # this is optional and can be removed
    base_set = set(base_classes)

    # here we iterate over sample labels and also get the correspondent sample index
    for sample_id, label in enumerate(dataset._labels):
        if label in base_set:
            base_categories_samples.append(sample_id)
        else:
            novel_categories_samples.append(sample_id)

    # here we create the dataset subsets
    # the torch Subset is just a wrapper around the dataset
    # it simply stores the subset indexes and the original dataset (your_subset.dataset)
    # when asking for sample i in the subset, torch will look for its original position in the dataset and retrieve it
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset
    base_dataset = torch.utils.data.Subset(dataset, base_categories_samples)
    novel_dataset = torch.utils.data.Subset(dataset, novel_categories_samples)
    return base_dataset, novel_dataset

def create_remapped_dataset(dataset, selected_classes):
    """Create a dataset subset with remapped labels.
    Args:
        dataset: Original dataset
        selected_classes: List of class indices to include
    Returns:
        subset dataset with labels remapped to [0, len(selected_classes)-1]
    """
    # Create mapping from original labels to new labels
    label_map = {old_label: new_label for new_label, old_label in enumerate(selected_classes)}
    selected_set = set(selected_classes)

    # Find samples and create new labels
    selected_samples = []
    new_labels = []

    for sample_id, label in enumerate(dataset._labels):
        if label in selected_set:
            selected_samples.append(sample_id)
            new_labels.append(label_map[label])

    # Create subset
    subset = torch.utils.data.Subset(dataset, selected_samples)

    # Add remapped labels to subset
    subset.remapped_labels = new_labels

    return subset

class RemappedDataset(torch.utils.data.Dataset):
    """Wrapper dataset that returns remapped labels"""
    def __init__(self, subset_dataset):
        self.dataset = subset_dataset.dataset
        self.indices = subset_dataset.indices
        self.labels = subset_dataset.remapped_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get original sample
        original_idx = self.indices[idx]
        image, _ = self.dataset[original_idx]  # Ignore original label

        # Return with remapped label
        return image, self.labels[idx]

## Baseline: CLIP Zero-Shot performance

We will now load the pretrained CLIP backbone and evaluate it on our dataset.

In [ ]:
# load CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/16", device=device) # preprocess contains CLIP's pre-defined augmentations

# define and inspect base and novel classes
_, _, tmp_test = get_data()
base_classes, novel_classes = base_novel_categories(tmp_test)
CLASS_NAMES = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
                "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
                "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
                "yellow iris", "globe-flower", "purple coneflower", "peruvian lily",
                "balloon flower", "giant white arum lily", "fire lily", "pincushion flower",
                "fritillary", "red ginger", "grape hyacinth", "corn poppy", "prince of wales feathers",
                "stemless gentian", "artichoke", "sweet william", "carnation", "garden phlox",
                "love in the mist", "mexican aster", "alpine sea holly", "ruby-lipped cattleya",
                "cape flower", "great masterwort", "siam tulip", "lenten rose", "barbeton daisy", "daffodil",
                "sword lily", "poinsettia", "bolero deep blue", "wallflower", "marigold", "buttercup",
                "oxeye daisy", "common dandelion", "petunia", "wild pansy", "primula", "sunflower",
                "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
                "pink-yellow dahlia", "cautleya spicata", "japanese anemone", "black-eyed susan",
                "silverbush", "californian poppy", "osteospermum", "spring crocus", "bearded iris",
                "windflower", "tree poppy", "gazania", "azalea", "water lily", "rose", "thorn apple",
                "morning glory", "passion flower", "lotus", "toad lily", "anthurium", "frangipani",
                "clematis", "hibiscus", "columbine", "desert-rose", "tree mallow", "magnolia", "cyclamen",
                "watercress", "canna lily", "hippeastrum", "bee balm", "ball moss", "foxglove",
                "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia", "blanket flower",
                "trumpet creeper", "blackberry lily"]
print("Base Class Names:", [(i, CLASS_NAMES[i]) for i in base_classes])
print("Novel Class Names:", [(i, CLASS_NAMES[i]) for i in novel_classes])

# get the three datasets
train_set, val_set, test_set = get_data(transform=preprocess)

# split classes into base and novel
base_classes, novel_classes = base_novel_categories(train_set)

# split the three datasets
train_base, _ = split_data(train_set, base_classes)
val_base, _ = split_data(val_set, base_classes)
test_base, test_novel = split_data(test_set, base_classes)

In [ ]:
# zero-shot predictions 

@torch.no_grad() # we don't want gradients
def eval(model, dataset, categories, batch_size, device, label=""):
    # let's set the model in evaluation mode
    model.eval()

    # Remap labels into a contiguous set starting from zero
    contig_cat2idx = {cat: idx for idx, cat in enumerate(categories)}

    # here we apply the standard CLIP template used for oxford flowers to all categories
    # and immediately tokenize each sentence (convert natural language into numbers - feel free to print the text input to inspect them)
    text_inputs = clip.tokenize(
        [f"a photo of a {CLASS_NAMES[c]}, a type of flower." for c in categories]
    ).to(device)

    # we can encode the text features once as they are shared for all images
    # therefore we do it outside the evaluation loop
    text_features = model.encode_text(text_inputs)
    # and here we normalize them (standard pratice with CLIP)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # simple dataloader creation
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # here we store the number of correct predictions we will make
    correct_predictions = 0
    for image, target in tqdm(dataloader, desc=label):
        # base categories range from 0 to 50, whil novel ones from 51 to 101
        # therefore we must map categories to the [0, 50], otherwise we will have wrong predictions
        # Map targets in contiguous set starting from zero
        # Labels needs to be .long() in pytorch
        target = torch.Tensor([contig_cat2idx[t.item()] for t in target]).long()

        image = image.to(device)
        target = target.to(device)

        # forward image through CLIP image encoder
        image_features = model.encode_image(image)
        # and normalize
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # here cosine similarity between image and text features and keep the argmax for every row (every image)
        predicted_class = (image_features @ text_features.T).argmax(dim=-1)
        # now we check which are correct, and sum them (False == 0, True == 1)
        correct_predictions += (predicted_class == target).sum().item()

    # and now we compute the accuracy
    accuracy = correct_predictions / len(dataset)
    return accuracy

base_accuracy = eval(model=model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Base Classes")
novel_accuracy = eval(model=model, dataset=test_novel, categories=novel_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Novel Classes")

print()
print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")
print(f"🔍 Novel classes accuracy: {novel_accuracy*100:.2f}%")

def harmonic_mean(base_accuracy, novel_accuracy):
    numerator = 2
    denominator = 1 / base_accuracy + 1 / novel_accuracy
    hm = numerator / denominator
    return hm

print(f"🔍 Harmonic Mean: {harmonic_mean(base_accuracy, novel_accuracy)*100:.2f}%")


## Double CLIP

One of the first ideas that came to mind was to enrich the prompt with context details related to the flower. So as first try in that direction we developd a systm that first used CLIP to infer the color of the given flower by classifying it within a determined range of colors, and then used the newfound information within the prompt for the specie classification.

In [ ]:
# code here

The approach resulted in no improvement over the zero-shot baseline, and only a bit of flection in performance results compared to zero-shot. Moreover the infernce time doubled compared to the baseline, due to the double use of clip. This issue could be solved by using a different model, or training one for color classification, but the idea was abandoned due to the fact that no improvement was fonud.

## CoOp

As next step, we decided to develop CoOp, following their paper, to then look for new original ideas/modifications that could lead us to better results. This approach aims at training a fixed amount of context token vectors that are fed to clip along with the classes names, allowing for a degree of freedom to push the right classes closer to the image vector represenations, to improve performance. During training, it minimizes prediction errors by using the cross-entropy loss with respect to the learnable context vectors while keeping the entire pre-trained parameters fixed. The gradients can be back-propagated all the way through the text encoder, distilling the rich knowledge encoded in the parameters for learning task-relevant context.

In [ ]:
# code here

This approach allowed us to grasp the strength of the context tokens that surround the class' ones; since it tackles all classes at once, it can lead to expect also an improvement on the novel classes, caused by the idea that the learned context is general enough to work with them too; however CoOp greatly improves performance on base classes but performes poorly on unseen classes, even much worse than the Zero-shot baseline, making it highly unsuitable for our objective, due to the high deficit on novel classes that we would have to regain.

## CoCoOp

If CoOp allowed to learn the context by learning a single representation acrossed all training data, CoCoOp provides a dual approach, learning to generate a perturbation conditioned by the current image along the initial learnable context, that influences the resulting text features for each single instance.
CoCoOp is implemented by instantiating a lightweight neural networks, called Meta-Net, on top of the M
context vectors, to generate for each input a conditional token (vector), which is then combined with the context vectors.


In [ ]:
# code here

In contrast with what can be one's initial beliefs, CoCoOp provided a slightly worse improvement on base classes compared to CoOp, while yielding a better performance on the unseen classes, striking a balance between the baseline and CoOp, but greatly improving the harmonic mean.
This is because instance-conditional context can generalize better because it shifts the focus away from a specific set of classes—for reducing overfitting—to each input instance, and hence to the entire task.

## MoCoOp

Our research for literature that allowed to improve base performance without drawbacks on the novel accuracy led us to MoCoOp: Mixture of Prompt Learning for Vision Language Models. Their model trains a gate-neural network, that selects the best expert among a given set. The expert consists on a specific prompt on which CoOp is applied, while using a custom loss for training. Moreover, the two most suitable experts are selected, and their embeddings are combined; the result is then used for classification.

In [ ]:
# code here

The model satisfies the performance constraints, improving base classes' performances while maintaining good results on the novel ones.

## KgCoOp

Implementing and analyzing MoCoOp sparked the idea that the performance result achieved by the model could be mostly attributed to the custom loss, which penalized the model the more it learned a representation that was shifting away from the original context feature. Therefore we decided to apply this idea to the original CoOp approach, without using multiple experts, having to train a selector, and merge together different inference results for the best experts, creating this way a lighter and more essential pipeline.

In [ ]:
# code here

Although KgCoOp exhibits really good results on base classes, two main issues refrained us from keeping it as our final main solution:
- Performance on novel classes was still affected by slightly degradation.
- Further research led us to the finding of KgCoOp, a paper that already explores the subject at matter, deeming our implementation not novel.

### Performance in-depth analysis - class name misalignment discovery
With no significant results at hand, we opted for analyzing the performance results across all models, hoping to find a lead for an area/feature in which the models were performing poorly. Calculating the accuracy per class showed that all models, zero-shot included, were able to perform almost error-free predictions for some classes, while they were not able to identify not even a single element, or barely a couple for others.

In [ ]:
# code here

We then proceeded by analyzing a couple of the problematic classes: with little research through wikipedia, we found other alternative names that could be used to identify the same species; as soon as we tried a couple of them we observed major improvements in the selected classes.

In [ ]:
# code here - single example of class, with wiki screenshot/link

## AKA Clip

The great improvements in the few manually analyzed classes prompted at the fact that CLIP, even in it's zero shot form, was already able to recognize and correcly classify the different flowers, but was doing so while using different names, that had a slightly different representation compared to the original ones.
To address the issue we implemented a slightly modified version of zero shot: we first collected multiple names for all classes in the dataset. Then all the different aliases were used concurrently in the classification phase, and only the best one was selected. The selected one was then mapped to the original label and compared with the ground truth.

In [ ]:
# code here

The results show major improvements in both base classes, where names were selected among the best performing ones, and novel classes, were we limited our work at name collection with no selection, fueling the hypotesis that using the right terms to identify the wanted class outweights the improvements given by the selection of the best context tokens that come along.